In [29]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path

import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [31]:
base_url = "http://personal-laptop.local:8080/"


In [33]:
script_dir = Path().resolve().parent

def create_url(base_url,first_part_endpoint,second_part_endpoint):
    return base_url + first_part_endpoint + second_part_endpoint
def getData(url):
    response = requests.get(url)

    # Check response status
    if response.status_code == 200:
        data = response.json()
        print(f"Received {len(data)} records.")
        return data
    else:
        print(f"Error: {response.status_code}")

def saveDataIntoDataFolder(url,data_file_name):
    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (data_file_name + ".json")
    
    data = getData(url)
    
    if data:
        with open(file_path, 'w') as file:
            json.dump(data, file)
        print(f"Data saved to {file_path}")
    else:
        print("No data to save.")


def loadDataFromFile(file_name):
    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None


In [34]:
first_part_endpoint = "dataAnalysisEndpoints/"
second_part_endpoint = "getAllUserInputsExperimentState"        

url = create_url(base_url,first_part_endpoint,second_part_endpoint)


saveDataIntoDataFolder(url,"UserInputs")

C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Received 271 records.
Data saved to C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\UserInputs.json


In [35]:
saveDataIntoDataFolder(url,"UserInputs")

C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Received 271 records.
Data saved to C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\UserInputs.json


In [36]:
df_userInputData = loadDataFromFile("UserInputs")

C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Loaded 271 records from C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\UserInputs.json


In [37]:
df_userInputData

,_id,experimentState,timestamp,userInputCategory,details
0,{'$oid': '6849ace703a598ff63a65bf8'},StartingExperiment,{'$date': 1749658855857},ExperimentState,NaN
1,{'$oid': '6849aefb03a598ff63a6667f'},InsertingSourcePollutant,{'$date': 1749659387774},ExperimentState,"{'are-doors-opened': 'on', 'are-people-inside'..."
2,{'$oid': '6849b77503a598ff63a68c90'},RemovingSourcePollutant,{'$date': 1749661557509},ExperimentState,NaN
3,{'$oid': '6849ba8d03a598ff63a69cd3'},InsertingSourcePollutant,{'$date': 1749662349567},ExperimentState,"{'are-doors-opened': 'on', 'are-people-inside'..."
4,{'$oid': '6849c2dc03a598ff63a6c299'},RemovingSourcePollutant,{'$date': 1749664476491},ExperimentState,NaN
...,...,...,...,...,...
266,{'$oid': '6877f07a3b4cde7d072ccdbb'},RemovingSourcePollutant,{'$date': 1752690810389},ExperimentState,NaN
267,{'$oid': '6877f2d73b4cde7d072cd2a3'},InsertingSourcePollutant,{'$date': 1752691415915},ExperimentState,"{'are-doors-opened': 'on', 'back-wall': '', 'f..."
268,{'$oid': '6877f42b3b4cde7d072cd4c1'},RemovingSourcePollutant,{'$date': 1752691755988},ExperimentState,NaN
269,{'$oid': '6877f95e3b4cde7d072cd5a9'},InsertingSourcePollutant,{'$date': 1752693086374},ExperimentState,"{'are-doors-opened': 'on', 'back-wall': '', 'f..."


In [38]:
if 'details' in df_userInputData.columns:
    details_df = df_userInputData["details"].apply(pd.Series)
    
    # Then join with the original DataFrame (drop the nested 'details' if desired)
    df_userInputData = pd.concat([df_userInputData.drop(columns=["details"]), details_df], axis=1)

In [39]:
df_userInputData.columns

Index([               '_id',    'experimentState',          'timestamp',
        'userInputCategory',                    0,   'are-doors-opened',
        'are-people-inside', 'are-windows-opened',         'front-wall',
                'item-used',              'notes',     'pollutant-type',
            'quantity-used',               'room',    'side-right-wall',
                'back-wall',     'side-left-wall'],
      dtype='object')

In [40]:
if 0 in df_userInputData.columns:
    df_userInputData =df_userInputData.drop(columns=[0])

In [42]:
# Convert epoch (assumes seconds) to datetime in local time
df_userInputData["epoch_ms"] = df_userInputData["timestamp"].apply(lambda x: x["$date"])
df_userInputData["timestamp_local"] = pd.to_datetime(df_userInputData["epoch_ms"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("Europe/Athens")
df_userInputData.drop(columns = ["timestamp"])

,_id,experimentState,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,back-wall,side-left-wall,epoch_ms,timestamp_local
0,{'$oid': '6849ace703a598ff63a65bf8'},StartingExperiment,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749658855857,2025-06-11 19:20:55.857000+03:00
1,{'$oid': '6849aefb03a598ff63a6667f'},InsertingSourcePollutant,ExperimentState,on,on,on,1.5,φαρμακευτική λοτιόν,,VOC,δύο κουταλιές της σούπας,Σαλόνι,1.7,NaN,NaN,1749659387774,2025-06-11 19:29:47.774000+03:00
2,{'$oid': '6849b77503a598ff63a68c90'},RemovingSourcePollutant,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749661557509,2025-06-11 20:05:57.509000+03:00
3,{'$oid': '6849ba8d03a598ff63a69cd3'},InsertingSourcePollutant,ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ,,VOC,τρεις κουταλιά της σούπας,Σαλόνι,1.7,NaN,NaN,1749662349567,2025-06-11 20:19:09.567000+03:00
4,{'$oid': '6849c2dc03a598ff63a6c299'},RemovingSourcePollutant,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749664476491,2025-06-11 20:54:36.491000+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
266,{'$oid': '6877f07a3b4cde7d072ccdbb'},RemovingSourcePollutant,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1752690810389,2025-07-16 21:33:30.389000+03:00
267,{'$oid': '6877f2d73b4cde7d072cd2a3'},InsertingSourcePollutant,ExperimentState,on,NaN,NaN,0.4,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,,,1.05,1752691415915,2025-07-16 21:43:35.915000+03:00
268,{'$oid': '6877f42b3b4cde7d072cd4c1'},RemovingSourcePollutant,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1752691755988,2025-07-16 21:49:15.988000+03:00
269,{'$oid': '6877f95e3b4cde7d072cd5a9'},InsertingSourcePollutant,ExperimentState,on,NaN,NaN,0.4,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,,,1.15,1752693086374,2025-07-16 22:11:26.374000+03:00


In [43]:
dimensions = ['front-wall','side-right-wall','back-wall','side-left-wall']
for dimension in dimensions:
    df_userInputData[dimension] = pd.to_numeric(df_userInputData[dimension], errors='coerce')

In [44]:
condition = df_userInputData["room"] == "Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55"

# Get the indices of the rows that match the condition
matching_indices = df_userInputData[condition].index

# Add the matching indices and the next row's indices
all_indices = list(matching_indices)

# Add the index of the next row for each match (if it's within bounds)
for idx in matching_indices:
    if idx + 1 < len(df_userInputData):
        all_indices.append(idx + 1)

# Get unique and sorted indices to avoid duplicates or order issues
all_indices = sorted(set(all_indices))

# Filter the dataframe
df_userInputData_smallScale = df_userInputData.loc[all_indices]

In [45]:
df_userInputData_smallScale = df_userInputData_smallScale.reset_index(drop=True) 
df_userInputData_smallScale

,_id,experimentState,timestamp,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,back-wall,side-left-wall,epoch_ms,timestamp_local
0,{'$oid': '686408456a2505a88d9b0665'},InsertingSourcePollutant,{'$date': 1751386181232},ExperimentState,on,NaN,NaN,0.95,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,0.95,1751386181232,2025-07-01 19:09:41.232000+03:00
1,{'$oid': '686409746a2505a88d9b087a'},RemovingSourcePollutant,{'$date': 1751386484742},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1751386484742,2025-07-01 19:14:44.742000+03:00
2,{'$oid': '68640eee885e900bf5302e85'},InsertingSourcePollutant,{'$date': 1751387886945},ExperimentState,on,NaN,NaN,0.95,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.05,1751387886945,2025-07-01 19:38:06.945000+03:00
3,{'$oid': '6864103c885e900bf5303149'},RemovingSourcePollutant,{'$date': 1751388220517},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1751388220517,2025-07-01 19:43:40.517000+03:00
4,{'$oid': '6864113e885e900bf530336f'},InsertingSourcePollutant,{'$date': 1751388478248},ExperimentState,on,NaN,NaN,0.95,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.05,1751388478248,2025-07-01 19:47:58.248000+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,{'$oid': '6877f07a3b4cde7d072ccdbb'},RemovingSourcePollutant,{'$date': 1752690810389},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1752690810389,2025-07-16 21:33:30.389000+03:00
174,{'$oid': '6877f2d73b4cde7d072cd2a3'},InsertingSourcePollutant,{'$date': 1752691415915},ExperimentState,on,NaN,NaN,0.40,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.05,1752691415915,2025-07-16 21:43:35.915000+03:00
175,{'$oid': '6877f42b3b4cde7d072cd4c1'},RemovingSourcePollutant,{'$date': 1752691755988},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1752691755988,2025-07-16 21:49:15.988000+03:00
176,{'$oid': '6877f95e3b4cde7d072cd5a9'},InsertingSourcePollutant,{'$date': 1752693086374},ExperimentState,on,NaN,NaN,0.40,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα id:1 Μ0.95 Α0.85 id:2 Μ0.95 Α1.55,NaN,NaN,1.15,1752693086374,2025-07-16 22:11:26.374000+03:00


In [46]:
df_userInputData_smallScale.loc[df_userInputData_smallScale['front-wall'] == 0.5, 'front-wall'] = 0.4


In [47]:
df_unique_position_count = df_userInputData_smallScale.groupby(['front-wall', 'side-right-wall','back-wall','side-left-wall'], dropna=False).size().reset_index(name='count')

In [48]:
df_unique_position_count

,front-wall,side-right-wall,back-wall,side-left-wall,count
0,0.40,NaN,NaN,0.55,1
1,0.40,NaN,NaN,0.95,4
2,0.40,NaN,NaN,1.05,4
3,0.40,NaN,NaN,1.15,4
4,0.40,NaN,NaN,1.25,4
5,0.40,NaN,NaN,1.35,3
6,0.40,NaN,NaN,1.75,2
7,0.95,NaN,NaN,0.55,2
8,0.95,NaN,NaN,0.95,6
9,0.95,NaN,NaN,1.05,6


In [ ]:
df_userInputData_smallScale_list_packed_by_position = 

In [49]:
print(df_unique_position_count["front-wall"].dtype)

float64


In [50]:
first_row= df_userInputData_smallScale.iloc[0]
last_row = df_userInputData_smallScale.iloc[-1]
start_time = first_row["epoch_ms"]
end_time   = last_row["epoch_ms"] 
print(first_row["timestamp_local"])
print(last_row["timestamp_local"])
first_part_endpoint = "timeSeriesEndpoints/"
second_part_endpoint = "getTimeSeriesData?start=" + str(start_time) + '&end=' +str(end_time)
file_name="TimeSeries"


2025-07-01 19:09:41.232000+03:00
2025-07-16 22:16:45.962000+03:00


In [51]:
url = create_url(base_url,first_part_endpoint,second_part_endpoint)
saveDataIntoDataFolder(url,file_name)
df_timeSeries = loadDataFromFile(file_name)

C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Received 126197 records.
Data saved to C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\TimeSeries.json
C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data
Loaded 126197 records from C:\Users\Andreas\Documents\PlatformIO\Projects\Diploma Project\dataAnalysis and machine learning\data\TimeSeries.json


In [52]:
df_timeSeries

,BME680:breathVocEquivalent,BME680:breathVocEquivalentAccuracy,BME680:co2Equivalent,BME680:co2EquivalentAccuracy,BME680:gasPercentage,BME680:gasPercentageAccuracy,BME680:gasResistance,BME680:humidity,BME680:iaq,BME680:iaqAccuracy,...,BME680:rawHumidity,BME680:rawTemperature,BME680:runInStatus,BME680:stabStatus,BME680:staticIaq,BME680:temperature,Id,Sensor,_id,timestamp
0,2.166788,0,1146.710,0,26.98844,0,77438.29,44.83818,87.30832,0,...,44.51159,34.35747,0,1,114.6710,34.24916,1,BME680,{'$oid': '686408466a2505a88d9b0672'},2025-07-01 16:09:42+00:00
1,2.167042,0,1146.762,0,27.97537,0,77656.13,44.64896,87.31131,0,...,44.35564,34.36766,0,1,114.6762,34.25935,1,BME680,{'$oid': '6864084a6a2505a88d9b0677'},2025-07-01 16:09:45+00:00
2,6.456104,3,1628.208,3,32.46553,3,174449.10,40.00833,103.36230,3,...,39.77594,35.94289,1,1,162.8208,35.83457,2,BME680,{'$oid': '6864084c6a2505a88d9b0682'},2025-07-01 16:09:43+00:00
3,6.464751,3,1628.799,3,32.48473,3,174266.30,40.01361,103.39340,3,...,39.77564,35.94033,1,1,162.8799,35.83202,2,BME680,{'$oid': '6864084c6a2505a88d9b0683'},2025-07-01 16:09:46+00:00
4,2.167497,0,1146.855,0,28.38902,0,78243.10,44.58836,87.31664,0,...,44.30568,34.37021,0,1,114.6855,34.26189,1,BME680,{'$oid': '6864084d6a2505a88d9b0686'},2025-07-01 16:09:48+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126192,1.972986,0,1105.388,0,42.64372,0,125000.00,41.91293,105.31080,0,...,41.65589,33.64655,0,1,110.5388,33.53823,1,BME680,{'$oid': '6877fa973b4cde7d072cd789'},2025-07-16 19:16:39+00:00
126193,6.955028,0,1661.038,0,59.09833,0,143195.50,41.14738,129.66920,0,...,40.91597,33.93308,0,1,166.1038,33.82477,2,BME680,{'$oid': '6877fa983b4cde7d072cd78c'},2025-07-16 19:16:40+00:00
126194,1.948035,0,1099.775,0,42.33880,0,125759.30,41.93336,104.79790,0,...,41.66766,33.64145,0,1,109.9775,33.53314,1,BME680,{'$oid': '6877fa9a3b4cde7d072cd78f'},2025-07-16 19:16:42+00:00
126195,7.145324,0,1672.942,0,59.46336,0,142337.30,41.16126,130.48610,0,...,40.92166,33.93053,0,1,167.2942,33.82222,2,BME680,{'$oid': '6877fa9b3b4cde7d072cd792'},2025-07-16 19:16:43+00:00


In [53]:
# 1. Select only the needed column
df_timeSeriesExperiments = df_timeSeries[['timestamp', 'Id', 'BME680:breathVocEquivalent']].copy()

In [54]:


# 2. Convert `timestamp` (UTC) to Europe/Athens tz-aware datetime
df_timeSeriesExperiments['timestamp'] = (
    pd.to_datetime(df_timeSeriesExperiments['timestamp'], utc=True)
      .dt.tz_convert('Europe/Athens')
)
# 3. Create the two “Id=…:” columns

df_timeSeriesExperiments['Id=1:BME680:breathVocEquivalent'] = np.where(
    df_timeSeriesExperiments['Id'] == 1, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
)
df_timeSeriesExperiments['Id=2:BME680:breathVocEquivalent'] = np.where(
    df_timeSeriesExperiments['Id'] == 2, df_timeSeriesExperiments['BME680:breathVocEquivalent'], np.nan
)

# 4. Set timestamp as the row index
#df_timeSeriesExperiments.set_index('timestamp', inplace=True)

# 5. Drop the old Id+raw measurement columns
df_timeSeriesExperiments = df_timeSeriesExperiments.drop(
    columns=['Id', 'BME680:breathVocEquivalent']
)

# Inspect result
df_timeSeriesExperiments.head()
df_timeSeriesExperiments

,timestamp,Id=1:BME680:breathVocEquivalent,Id=2:BME680:breathVocEquivalent
0,2025-07-01 19:09:42+03:00,2.166788,NaN
1,2025-07-01 19:09:45+03:00,2.167042,NaN
2,2025-07-01 19:09:43+03:00,NaN,6.456104
3,2025-07-01 19:09:46+03:00,NaN,6.464751
4,2025-07-01 19:09:48+03:00,2.167497,NaN
...,...,...,...
126192,2025-07-16 22:16:39+03:00,1.972986,NaN
126193,2025-07-16 22:16:40+03:00,NaN,6.955028
126194,2025-07-16 22:16:42+03:00,1.948035,NaN
126195,2025-07-16 22:16:43+03:00,NaN,7.145324


In [56]:
df_timeSeriesExperiments["timestamp"].dtype

datetime64[ns, Europe/Athens]

In [ ]:
num_experiments = df_timeSeriesExperiments.shape[0] // 2
fig, axes = plt.subplots(num_experiments, 1, figsize=(14, 2 * num_experiments), sharex=False)
df_userInputData_smallScale


for i in range(0, len(df_userInputData_smallScale), 2):
    inserting_row = df_userInputData_smallScale.iloc[i]
    removing_row = df_userInputData_smallScale.iloc[i+1]
    data = df_userInputData_smallScale[(df_userInputData_smallScale['timestamp'] >= inserting_row["timestamp"]) & (df_userInputData_smallScale['timestamp'] <= removing_row["timestamp"])]
    # RIGHT PLOT: Id=1 and Id=2 for BME680
    sns.lineplot(
        data=data, 
        x="timestamp",
        y='Id=1:BME680:breathVocEquivalent',
        label="Id=1",
        marker='o'
    )
    sns.lineplot(
        data=data,
        x="timestamp",
        y='Id=2:BME680:breathVocEquivalent',
        label="Id=2",
        marker='s'
    )
    axes[(i // 2)].set_title(f"BME680:breathVocEquivalent at { inserting_row['front-wall'] } and { inserting_row['side-left-wall'] }")
    axes[(i // 2)].set_xlabel("Timestamp")
    axes[(i // 2)].set_ylabel("VOC")
    axes[(i // 2)].legend()

# Adjust layout
plt.tight_layout()
plt.show()